In [ ]:
import argparse
import glob
import os
import sys

import numpy as np
import pandas as pd

In [ ]:
NK = 23
TRACKS = (1, 103)         
TRACK_TOL = 0.05          
MIN_CLEAN_FRAC = 0.5     
MIN_NS1 = 0.5             
BASE = (2016, 2025)
MIN_BASE_N = 6
COORD_TOL = 1e-4         
S1_VARS = ["vv_db", "vh_db", "cr_db", "rvi"]

In [ ]:
def read_table(d, stem):
    pq, csv = os.path.join(d, stem + ".parquet"), os.path.join(d, stem + ".csv")
    df = pd.read_parquet(pq) if os.path.exists(pq) else pd.read_csv(csv, dtype={"pid": str})
    df["pid"] = df["pid"].astype(str)
    return df


def load_s1(s1_dir):
    files = sorted(glob.glob(os.path.join(s1_dir, "s1_*.csv")))
    if not files:
        raise FileNotFoundError(f"No s1_*.csv in {s1_dir}")
    df = pd.concat((pd.read_csv(f, dtype={"pid": str}) for f in files), ignore_index=True)
    long = pd.wide_to_long(df, stubnames=["vv_k", "vh_k", "ns1_k", "orb_k"],
                           i=["pid", "year", "lon", "lat"], j="k").reset_index()
    long = long.rename(columns={"vv_k": "vv", "vh_k": "vh", "ns1_k": "ns1", "orb_k": "orb"})
    long["k"] = long["k"].astype(int)
    print(f"S1: {len(files)} files, {long['pid'].nunique()} points, years {long['year'].min()}-{long['year'].max()}, "
          f"windows {long['k'].min()}-{long['k'].max()}")
    return long


def check_ids(s1, feat):
    f = feat.drop_duplicates("pid")[["pid", "lon", "lat"]]
    s = s1.drop_duplicates("pid")[["pid", "lon", "lat"]]
    m = s.merge(f, on="pid", how="left", suffixes=("_s1", "_f"))
    missing = m["lon_f"].isna().mean()
    bad = ((m["lon_s1"] - m["lon_f"]).abs() > COORD_TOL) | ((m["lat_s1"] - m["lat_f"]).abs() > COORD_TOL)
    bad = bad & m["lon_f"].notna()
    print(f"pid check: {missing:.1%} of S1 pids absent from the feature table; "
          f"{bad.mean():.1%} have different coordinates (tolerance {COORD_TOL} deg)")
    if missing > 0.01 or bad.mean() > 0.01:
        raise ValueError("S1 pids do not point to the same places as the feature table. "
                        )


In [ ]:
def clean_tracks(s1, tracks=TRACKS):
    o = s1["orb"]
    near = pd.DataFrame({t: (o - t).abs() < TRACK_TOL for t in tracks})
    s1["pure_track"] = np.select([near[t] for t in tracks], list(tracks), default=np.nan)
    dom = s1.dropna(subset=["pure_track"]).groupby("pid")["pure_track"].agg(lambda x: x.mode().iloc[0])
    s1["dom_track"] = s1["pid"].map(dom)
    has = s1["vv"].notna()
    s1["clean"] = has & (s1["pure_track"] == s1["dom_track"]) & (s1["ns1"] >= MIN_NS1)
    frac = s1[has].groupby("pid")["clean"].mean()
    keep = set(frac[frac >= MIN_CLEAN_FRAC].index)
    n_all = s1["pid"].nunique()
    print(f"\nTrack handling: {len(keep)} / {n_all} points kept for S1 "
          f"({n_all - len(keep)} dropped as track-overlap points)")
    print("Dominant track of kept points:\n", dom[dom.index.isin(keep)].value_counts().to_string())
    print(f"Share of windows with data that are clean (kept points): "
          f"{s1[has & s1['pid'].isin(keep)]['clean'].mean():.3f}")
    s1["use"] = s1["clean"] & s1["pid"].isin(keep)
    return s1


def derive(s1):
    ok = s1["use"]
    vv = s1["vv"].where(ok)
    vh = s1["vh"].where(ok)
    s1["vv_db"] = 10 * np.log10(vv)
    s1["vh_db"] = 10 * np.log10(vh)
    s1["cr_db"] = s1["vh_db"] - s1["vv_db"]
    s1["rvi"] = 4 * vh / (vv + vh)
    return s1


In [ ]:
def zscore_loyo(df, col, keys=("pid", "k"), base=BASE, loyo=True, min_n=MIN_BASE_N):
    keys = list(keys)
    y0, y1 = base
    x = df[col].astype(float)
    inb = df["year"].between(y0, y1) & x.notna()
    tmp = df[keys].copy()
    tmp["x"] = x.where(inb)
    tmp["x2"] = tmp["x"] ** 2
    g = tmp.groupby(keys)["x"].agg(["count", "sum"]).join(tmp.groupby(keys)["x2"].sum().rename("ss"))
    m = df[keys].join(g, on=keys)
    n, s, ss = (m[c].astype(float) for c in ["count", "sum", "ss"])
    if loyo:
        n = n - inb.astype(float)
        s = s - x.where(inb, 0.0)
        ss = ss - (x ** 2).where(inb, 0.0)
    mu = s / n
    sd = np.sqrt(np.clip((ss - n * mu ** 2) / (n - 1), 0, None))
    z = (x - mu) / np.maximum(sd, 0.05 * np.nanstd(x))
    return z.where(n >= min_n)


In [ ]:
# main
def save_table(df, path_no_ext):
    try:
        df.to_parquet(path_no_ext + ".parquet", index=False)
        print(f"Saved {path_no_ext}.parquet")
    except Exception as e:
        df.to_csv(path_no_ext + ".csv", index=False)
        print(f"parquet failed ({e}); saved {path_no_ext}.csv")


def sanity_vs_label(feat_s1, label_dir, max_lag=3):
    lab = read_table(label_dir, "label_long")[["pid", "year", "k", "z"]].dropna()
    lab["t"] = lab["year"] * NK + lab["k"]
    f = feat_s1[["pid", "year", "k", "vv_z", "vh_z", "cr_z", "rvi_z"]].copy()
    f["t"] = f["year"] * NK + f["k"]
    rows = []
    for h in range(0, max_lag + 1):
        g = f.copy()
        g["t"] = g["t"] + h                        # S1 at t-h lines up with the label at t
        m = lab.merge(g.drop(columns=["year", "k"]), on=["pid", "t"], how="inner")
        for c in ["vv_z", "vh_z", "cr_z", "rvi_z"]:
            rows.append(dict(lead_days=16 * h, feature=c, r=m[["z", c]].corr().iloc[0, 1], n=int(m[c].notna().sum())))
    out = pd.DataFrame(rows).pivot(index="lead_days", columns="feature", values="r")
    print("\nSanity: point-level correlation of S1 anomaly (at t-h) with NDVI z at t "
          "(n is inflated by shared pixels; just a smell test):\n", out.round(3).to_string())


def run(s1_dir, feat_dir, out_dir, label_dir=None, tracks=TRACKS):
    os.makedirs(out_dir, exist_ok=True)
    feat = read_table(feat_dir, "features_long")
    s1 = load_s1(s1_dir)
    check_ids(s1, feat)
    s1 = derive(clean_tracks(s1, tracks))
    for c in S1_VARS:
        s1[c.replace("_db", "") + "_z" if c.endswith("_db") else c + "_z"] = zscore_loyo(s1, c)
    print("\nValid share by year (kept points):")
    print(s1.groupby("year")["use"].mean().round(3).to_string())
    keep_cols = ["pid", "year", "k", "vv_db", "vh_db", "cr_db", "rvi", "vv_z", "vh_z", "cr_z", "rvi_z", "ns1"]
    s1o = s1[keep_cols].rename(columns={"ns1": "s1_n"})
    for c in s1o.columns[3:]:
        s1o[c] = s1o[c].astype("float32")
    # only keep rows with data: keeps the merge light
    s1o = s1o[s1o["vv_db"].notna()]
    out = feat.merge(s1o, on=["pid", "year", "k"], how="left")
    save_table(out, os.path.join(out_dir, "features_long_s1"))
    if label_dir:
        sanity_vs_label(out, label_dir)
    return out


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--s1-dir", required=True)
    ap.add_argument("--feat-dir", required=True, help="folder with features_long from 03")
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--label-dir", default=None, help="optional: print a sanity correlation with the NDVI label")
    ap.add_argument("--tracks", nargs="*", type=int, default=list(TRACKS))
    a = ap.parse_args(argv)
    return run(a.s1_dir, a.feat_dir, a.out_dir, a.label_dir, tuple(a.tracks))


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [2]:
main(["--s1-dir", r"crop_stress/s1_v3/", "--feat-dir", r"crop_stress\features",
      "--out-dir", r"crop_stress\features", "--label-dir", r"crop_stress\label"])

S1: 10 files, 3185 points, years 2016-2025, windows 0-12
pid check: 0.0% of S1 pids absent from the feature table; 0.0% have different coordinates (tolerance 0.0001 deg)

Track handling: 2552 / 3185 points kept for S1 (633 dropped as track-overlap points)
Dominant track of kept points:
 pure_track
1.0      2170
103.0     382
Share of windows with data that are clean (kept points): 1.000

Valid share by year (kept points):
year
2016    0.749
2017    0.801
2018    0.801
2019    0.801
2020    0.801
2021    0.738
2022    0.801
2023    0.801
2024    0.730
2025    0.801
Saved crop_stress\features\features_long_s1.parquet

Sanity: point-level correlation of S1 anomaly (at t-h) with NDVI z at t (n is inflated by shared pixels; just a smell test):
 feature     cr_z  rvi_z   vh_z   vv_z
lead_days                            
0          0.279  0.281  0.380  0.303
16         0.210  0.210  0.271  0.209
32         0.162  0.161  0.214  0.170
48         0.133  0.131  0.159  0.120


,pid,year,lon,lat,k,chirps,dry,sm1,sm2,t2m,...,spi96,vv_db,vh_db,cr_db,rvi,vv_z,vh_z,cr_z,rvi_z,s1_n
0,105,2001,2.737942,6.969793,0,3.497316,15,0.107035,0.094344,27.424453,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,105,2001,2.737942,6.969793,1,0.000000,16,0.092514,0.087558,28.022861,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,105,2001,2.737942,6.969793,2,5.478588,15,0.071417,0.083159,28.377664,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,105,2001,2.737942,6.969793,3,16.375793,12,0.103606,0.080867,28.903681,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,105,2001,2.737942,6.969793,4,24.936403,8,0.196195,0.121286,27.980042,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1831370,8385,2025,3.306606,8.311457,18,61.393717,9,0.354122,0.366390,26.721655,...,-0.477917,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1831371,8385,2025,3.306606,8.311457,19,16.120694,13,0.290613,0.298242,27.221108,...,-0.480608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1831372,8385,2025,3.306606,8.311457,20,3.000568,16,0.234038,0.258766,28.081444,...,0.280164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1831373,8385,2025,3.306606,8.311457,21,10.832093,12,0.208647,0.234236,28.245163,...,0.812322,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
main(["--feat-dir", r"D:\crop_stress\features", "--feat-file", "features_long_s1",
      "--label-dir", r"D:\crop_stress\label", "--out-dir", r"D:\crop_stress\models_s1",
      "--sets", "ndvi", "climate+ndvi", "ndvi+s1", "climate+ndvi+s1",
      "--models", "xgb", "--min-year", "2016", "--year-groups", "5"])